# DeepInteractome: Exploratory Data Analysis (EDA)

This notebook explores the processed genomic features (`data/processed/genomic_features.csv`) used to train our DeepInteractome models. We'll visualize feature distributions, examine class balance, and look for correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load the Data

First, we load the engineered features.

In [ ]:
# Load data
data_path = '../../data/processed/genomic_features.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully. Shape: {df.shape}")
    display(df.head())
else:
    print(f"Error: File not found at {data_path}. Please run the data download and processing scripts first.")

## 2. Target Variable Distribution

Let's look at the balance of our target classes (`Target_Label`: 1 for Pathogenic, 0 for Benign).

In [ ]:
if 'Target_Label' in df.columns:
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(data=df, x='Target_Label')
    plt.title('Distribution of Pathogenic vs Benign Variants')
    plt.xlabel('Target Label (0 = Benign, 1 = Pathogenic)')
    plt.ylabel('Count')
    
    # Add counts on top of bars
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.show()
    
    # Print percentages
    counts = df['Target_Label'].value_counts(normalize=True) * 100
    print(f"Benign (0): {counts[0]:.2f}%")
    print(f"Pathogenic (1): {counts[1]:.2f}%")
else:
    print("Target_Label column not found.")

## 3. Feature Distributions

Let's examine some key numerical tabular features like Allele Frequency (`AF`) or Position (`POS`).

In [ ]:
if 'AF' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x='AF', hue='Target_Label', element='step', stat='density', common_norm=False)
    plt.title('Allele Frequency (AF) Distribution by Class')
    plt.xlabel('Allele Frequency')
    plt.ylabel('Density')
    plt.xlim(0, 0.05) # Zoom in on rare variants
    plt.show()
else:
    print("AF column not found.")

## 4. Correlation Analysis

Visualizing the correlation between numerical features can help identify redundant information or strong predictors.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude some categorical/one-hot columns for clarity if there are too many
cols_to_plot = [c for c in numeric_cols if not c.startswith('Ref_') and not c.startswith('Alt_') and not c.startswith('Up_') and not c.startswith('Down_')]

if len(cols_to_plot) > 1:
    corr = df[cols_to_plot].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Correlation Matrix of Key Features')
    plt.show()
else:
    print("Not enough numeric columns for correlation matrix.")